<a href="https://colab.research.google.com/github/yccccc12/DocStruct-VLM/blob/master/experiments/MonkeyOCR_pro_3B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:
!cat /etc/*release

DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


In [3]:
!apt-get --purge remove "*cublas*" "*cuda*"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'cublasmp-cuda-11' for glob '*cublas*'
Note, selecting 'cublasmp-cuda-12' for glob '*cublas*'
Note, selecting 'cublasmp-cuda-13' for glob '*cublas*'
Note, selecting 'cublasmp0' for glob '*cublas*'
Note, selecting 'libcublasmp0-dev-cuda-11' for glob '*cublas*'
Note, selecting 'libcublasmp0-dev-cuda-12' for glob '*cublas*'
Note, selecting 'libcublasmp0-dev-cuda-13' for glob '*cublas*'
Note, selecting 'libcublasmp0-cuda-11' for glob '*cublas*'
Note, selecting 'libcublasmp0-cuda-12' for glob '*cublas*'
Note, selecting 'libcublasmp0-cuda-13' for glob '*cublas*'
Note, selecting 'libcublas.so.11' for glob '*cublas*'
Note, selecting 'libcublas.so.12' for glob '*cublas*'
Note, selecting 'libcublas.so.13' for glob '*cublas*'
Note, selecting 'libcublas-dev-11-0' for glob '*cublas*'
Note, selecting 'libcublas-dev-11-1' for glob '*cublas*'
Note, selecting 'libcublas-dev-11-7' for glob '*

In [4]:
!apt-get autoremove -y

# Download the CUDA 12.6 runfile
!wget https://developer.download.nvidia.com/compute/cuda/12.6.0/local_installers/cuda_12.6.0_560.28.03_linux.run

!sudo sh cuda_12.6.0_560.28.03_linux.run --silent --toolkit

!ln -sf /usr/local/cuda-12.6 /usr/local/cuda
import os
os.environ['PATH'] += ':/usr/local/cuda-12.6/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda-12.6/lib64'

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
--2026-03-05 14:51:42--  https://developer.download.nvidia.com/compute/cuda/12.6.0/local_installers/cuda_12.6.0_560.28.03_linux.run
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.59.88.3, 23.59.88.13, 23.59.88.4, ...
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.59.88.3|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4333105923 (4.0G) [application/octet-stream]
Saving to: ‘cuda_12.6.0_560.28.03_linux.run’

cuda_12.6.0_560.28. 100%[===================>]   4.04G  36.4MB/s    in 87s     

2026-03-05 14:53:09 (47.6 MB/s) - ‘cuda_12.6.0_560.28.03_linux.run’ saved [4333105923/4333105923]



In [5]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Fri_Jun_14_16:34:21_PDT_2024
Cuda compilation tools, release 12.6, V12.6.20
Build cuda_12.6.r12.6/compiler.34431801_0


In [7]:
!git clone https://github.com/Yuliang-Liu/MonkeyOCR.git
%cd MonkeyOCR

!pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install paddlex[base]==3.3.0
!pip install langchain==0.3.26

Cloning into 'MonkeyOCR'...
remote: Enumerating objects: 1220, done.
remote: Counting objects: 100% (520/520), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 1220 (delta 417), reused 366 (delta 366), pack-reused 700 (from 4)
Receiving objects: 100% (1220/1220), 18.52 MiB | 39.75 MiB/s, done.
Resolving deltas: 100% (722/722), done.
/content/MonkeyOCR
Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu126/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.3 MB/s et

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.16
    Uninstalling langchain-core-1.2.16:
      Successfully uninstalled langchain-core-1.2.16
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.1
    Uninstalling langchain-text-splitters-1.1.1:
      Successfully uninstalled langchain-text-splitters-1.1.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.10
    Uninstalling langchain-1.2.10:
      Successfully uninstalled langchain-1.2.10
ERROR: pip's dependency r

In [8]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu126
!pip install -e .
!pip install lmdeploy==0.9.2

Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.4/764.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.7/166.7 MB 6.4 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.25.1
    Uninstalling nvidia-nccl-cu12-2.25.1:
      Successfully uninst

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.8/452.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 30.1 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully un

In [9]:
!pip install huggingface_hub

!python tools/download_model.py -n MonkeyOCR-pro-3B

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 23 files:   0% 0/23 [00:00<?, ?it/s]
config.json: 1.24kB [00:00, 2.67MB/s]

Recognition/model-00001-of-00002.safeten(…):   0% 0.00/5.00G [00:00<?, ?B/s]

added_tokens.json: 100% 605/605 [00:00<00:00, 2.94MB/s]


merges.txt: 0.00B [00:00, ?B/s

In [10]:
# 110s
!python parse.py "/content/form_test_1.png"

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/content/MonkeyOCR/magic_pdf/model/batch_analyze_llm.py:175: SyntaxWarning: invalid escape sequence '\$'
  return output.replace('$$', '\$\$').strip('$').strip()
2026-03-05 15:18:24.320117: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772723904.532864    8021 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772723904.594366    8021 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cu

In [13]:
!zip -r /content/file.zip /content/MonkeyOCR/output/form_test_1


  adding: content/MonkeyOCR/output/form_test_1/ (stored 0%)
  adding: content/MonkeyOCR/output/form_test_1/form_test_1.md (deflated 35%)
  adding: content/MonkeyOCR/output/form_test_1/form_test_1_layout.pdf (deflated 12%)
  adding: content/MonkeyOCR/output/form_test_1/form_test_1_model.pdf (deflated 13%)
  adding: content/MonkeyOCR/output/form_test_1/form_test_1_middle.json (deflated 94%)
  adding: content/MonkeyOCR/output/form_test_1/form_test_1_content_list.json (deflated 80%)
  adding: content/MonkeyOCR/output/form_test_1/form_test_1_spans.pdf (deflated 6%)
  adding: content/MonkeyOCR/output/form_test_1/images/ (stored 0%)


In [14]:
# 86s
!python parse.py "/content/handwritten_text_test_1.png"

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-03-05 15:27:27.541703: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772724447.569240   15859 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772724447.577309   15859 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772724447.608181   15859 computation_placer.cc:177] computation placer already registered. Please check lin

In [15]:
!zip -r /content/file_2.zip /content/MonkeyOCR/output/handwritten_text_test_1


  adding: content/MonkeyOCR/output/handwritten_text_test_1/ (stored 0%)
  adding: content/MonkeyOCR/output/handwritten_text_test_1/handwritten_text_test_1_middle.json (deflated 91%)
  adding: content/MonkeyOCR/output/handwritten_text_test_1/handwritten_text_test_1_layout.pdf (deflated 0%)
  adding: content/MonkeyOCR/output/handwritten_text_test_1/handwritten_text_test_1_model.pdf (deflated 0%)
  adding: content/MonkeyOCR/output/handwritten_text_test_1/handwritten_text_test_1_spans.pdf (deflated 0%)
  adding: content/MonkeyOCR/output/handwritten_text_test_1/handwritten_text_test_1.md (deflated 30%)
  adding: content/MonkeyOCR/output/handwritten_text_test_1/handwritten_text_test_1_content_list.json (deflated 53%)
  adding: content/MonkeyOCR/output/handwritten_text_test_1/images/ (stored 0%)


In [16]:
# 70s
!python parse.py "/content/utar_form_digital.pdf"

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-03-05 15:30:55.576093: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772724655.604278   22101 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772724655.612477   22101 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772724655.643933   22101 computation_placer.cc:177] computation placer already registered. Please check lin

In [17]:
!zip -r /content/file_3.zip /content/MonkeyOCR/output/utar_form_digital


  adding: content/MonkeyOCR/output/utar_form_digital/ (stored 0%)
  adding: content/MonkeyOCR/output/utar_form_digital/utar_form_digital_middle.json (deflated 93%)
  adding: content/MonkeyOCR/output/utar_form_digital/utar_form_digital_content_list.json (deflated 72%)
  adding: content/MonkeyOCR/output/utar_form_digital/utar_form_digital_model.pdf (deflated 12%)
  adding: content/MonkeyOCR/output/utar_form_digital/utar_form_digital_spans.pdf (deflated 11%)
  adding: content/MonkeyOCR/output/utar_form_digital/utar_form_digital.md (deflated 55%)
  adding: content/MonkeyOCR/output/utar_form_digital/images/ (stored 0%)
  adding: content/MonkeyOCR/output/utar_form_digital/images/b865353ae21c49291ead789f840256fb194b6992ee8e208146ad9d3d20a3250b.jpg (deflated 18%)
  adding: content/MonkeyOCR/output/utar_form_digital/images/fa02638809924f6b4de3656fe44fa7693b3b10a9facd9ab50b6297384d072e7c.jpg (deflated 14%)
  adding: content/MonkeyOCR/output/utar_form_digital/images/ae8dc99f955b579c0bbc38929263c5

In [18]:
# 154s
!python parse.py "/content/Test1.png"

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-03-05 15:34:37.957638: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772724877.996713   28405 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772724878.005805   28405 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772724878.037022   28405 computation_placer.cc:177] computation placer already registered. Please check lin

In [19]:
!zip -r /content/file_4.zip /content/MonkeyOCR/output/Test1


  adding: content/MonkeyOCR/output/Test1/ (stored 0%)
  adding: content/MonkeyOCR/output/Test1/Test1_middle.json (deflated 92%)
  adding: content/MonkeyOCR/output/Test1/Test1_layout.pdf (deflated 3%)
  adding: content/MonkeyOCR/output/Test1/Test1.md (deflated 70%)
  adding: content/MonkeyOCR/output/Test1/Test1_spans.pdf (deflated 3%)
  adding: content/MonkeyOCR/output/Test1/images/ (stored 0%)
  adding: content/MonkeyOCR/output/Test1/images/cc2487121ce887b9dc096dd4882336fbb1417a329fcc52e4df50a5c79bc1c013.jpg (deflated 4%)
  adding: content/MonkeyOCR/output/Test1/Test1_content_list.json (deflated 71%)
  adding: content/MonkeyOCR/output/Test1/Test1_model.pdf (deflated 5%)


In [20]:
# 95s
!python parse.py "/content/UECM3993_MAY2025-1-3.pdf"

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-03-05 15:37:32.812585: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772725052.838914   34497 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772725052.847467   34497 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772725052.878283   34497 computation_placer.cc:177] computation placer already registered. Please check lin

In [21]:
!zip -r /content/file_5.zip /content/MonkeyOCR/output/UECM3993_MAY2025-1-3


  adding: content/MonkeyOCR/output/UECM3993_MAY2025-1-3/ (stored 0%)
  adding: content/MonkeyOCR/output/UECM3993_MAY2025-1-3/UECM3993_MAY2025-1-3_middle.json (deflated 94%)
  adding: content/MonkeyOCR/output/UECM3993_MAY2025-1-3/UECM3993_MAY2025-1-3_spans.pdf (deflated 26%)
  adding: content/MonkeyOCR/output/UECM3993_MAY2025-1-3/UECM3993_MAY2025-1-3.md (deflated 54%)
  adding: content/MonkeyOCR/output/UECM3993_MAY2025-1-3/UECM3993_MAY2025-1-3_layout.pdf (deflated 29%)
  adding: content/MonkeyOCR/output/UECM3993_MAY2025-1-3/UECM3993_MAY2025-1-3_model.pdf (deflated 30%)
  adding: content/MonkeyOCR/output/UECM3993_MAY2025-1-3/UECM3993_MAY2025-1-3_content_list.json (deflated 73%)
  adding: content/MonkeyOCR/output/UECM3993_MAY2025-1-3/images/ (stored 0%)
  adding: content/MonkeyOCR/output/UECM3993_MAY2025-1-3/images/71d391fc6a542ac9261a4a972deeaf4e20ff9c5ee368eb0b16ab92eef10fdc00.jpg (deflated 3%)


## References
MonkeyOCR: https://github.com/Yuliang-Liu/MonkeyOCR
\
Cuda126: https://developer.nvidia.com/cuda-12-6-0-download-archive?target_os=Linux&target_arch=x86_64&Distribution=Ubuntu&target_version=22.04&target_type=runfile_local